In [ ]:
cd(joinpath(@__DIR__, ".."))   # run from the repository root

# Two Hodge stars for primal 1-forms — sandwich vs lift

Both are built from the **same** stencil, the **same** basis $(x-x_0)^\alpha$ and the
**same** pseudo-inverse $\Pi_e=\mathrm{pinv}(A_e)$, reusing `integrate_basic_fields_*_edge`,
`get_neighbors_*_primal` and `edge_integral` from `juliaDEC.jl` verbatim.

|  | **sandwich** $\Pi'M\Pi+\tau\,\mathrm{stab}$ | **lift** $\tilde R_1(\star_K\,\cdot)\circ\Pi$ |
|---|---|---|
| what it is | a bilinear **form** | a **map** |
| symmetry | exact, PSD by construction | none |
| energy exact on $P_k$ | yes | no |
| star property $H\mathcal R_1u=\tilde{\mathcal R}_1(\star u)$ | **no — $O(1)$, flat in $h$** | **yes, exactly** |
| use for | $L=d_0'Hd_0$, energies, eigenvalues | fluxes, the mixed system |

`hodge_star_lift_operator(...; K=nothing, rescale=false)` reproduces your
`hodge_star_lsq_operator` identically.

**Road map**

1. mesh hypotheses the construction actually needs (orientation, well-centredness, tiling)
2. a picture of one diamond and its stencil
3. $\Pi_eA_e=\mathrm{Id}$ — the single algebraic fact both theorems rest on
4. build both stars; symmetry
5. **the star property** — the decisive test, and that it does not improve with $h$
6. **eigenvalues** — why `only_linear=true` gives you garbage and `false` does not
7. Poisson convergence
8. flux, isolated from the solve
9. the mixed (Hirani) system
10. `report(mesh)` — all of the above in one call

## 0. Setup

In [ ]:
using VoronoiMeshes, TensorsLite, TensorsLiteGeometry, SmallCollections, LinearAlgebra, NCDatasets
using StaticArrays, SparseArrays, QuadGK
using CairoMakie, LaTeXStrings
using LowRankApprox

# Same TensorsLite/Makie workaround as whitney_basis.ipynb: TensorsLite pirates
# similar(::Broadcasted, ::Type{T<:AbstractTensor}) and that also catches T === Union{}.
Base.similar(bc::Base.Broadcast.Broadcasted, ::Type{Union{}}) =
    Array{Union{}}(undef, length.(axes(bc)))

include("src/juliaDEC.jl");
include("src/hodge_stars.jl");

In [ ]:
mesh = VoronoiMesh("meshes/mesh2_regular.nc")

println("cells (Delaunay vertices) = ", mesh.cells.n)
println("edges                     = ", mesh.edges.n)
println("triangles                 = ", mesh.vertices.n)
println("period                    = ", mesh.x_period, " x ", mesh.y_period)
println("h = max |e|               = ", maximum(edge_lengths(mesh, true)))
plotmesh(mesh)

## 1. Does this mesh satisfy what the construction assumes?

Three things, and all three are load-bearing.

**Orientation.** Everything assumes `mesh.edges.vertices` is ordered so that
$t_{\star e}=R(t_e)$ with $R(w)=(-w_2,w_1)$ — the MPAS convention, and precisely what
makes `hodge_applied=true` the *discrete star*. Violate it on some edges and the lift
picks up $O(1)$ antisymmetry, the sandwich star defect jumps from a few percent to
$O(1)$, and $d_0'Hd_0$ fills with large negative eigenvalues. (I hit exactly this while
checking the code, which is why there is a function for it.)

**Well-centredness.** Delaunay gives an *empty circumcircle*, not an *interior*
circumcentre. On an obtuse triangle the circumcentre crosses the edge, the diamond
becomes a bow tie, and the two triangles double-count instead of tiling.

**Tiling.** $\sum_e|D_e|=|\Omega|$ is the identity that makes the sandwich assembly a
valid approximation of $\int_\Omega u\cdot Ku$. If it fails, no energy statement holds.

In [ ]:
println("dual edges mis-oriented : ", check_dual_orientation(mesh), "    <- must be 0")

tot, om, nbad = check_diamond_partition(mesh)
println("sum |D_e|               : ", tot)
println("|Omega|                 : ", om, "     rel err ", abs(tot - om)/om)
println("badly-centred edges     : ", nbad, "    <- must be 0")

## 2. The object itself: one diamond, one stencil

$D_e=\mathrm{conv}(e\,,\star e)$ — red is the Delaunay edge, blue its Voronoi dual,
orange the diamond the mass matrix integrates over, grey the stencil the reduction
matrix is built from.

$[P_1]^2$ (6 unknowns) uses the 13-point stencil; $[P_2]^2$ (12 unknowns) the 19-point one.

In [ ]:
e  = 7
x0 = mesh.edges.position[e]
c1, c2 = edge_endpoints(mesh, e, true)      # Delaunay edge
v1, v2 = edge_endpoints(mesh, e, false)     # Voronoi edge

fig = Figure(size = (820, 400))
for (col, S, name) in ((1, get_neighbors_2level_primal(mesh, e), "[P1]^2  -  13-point"),
                       (2, get_neighbors_3level_primal(mesh, e), "[P2]^2  -  19-point"))
    ax = Axis(fig[1, col], aspect = DataAspect(),
              title = name * "   (|S| = " * string(length(S)) * ")")
    for m in S
        a, b = edge_endpoints(mesh, m, true; about = x0)
        lines!(ax, [a[1], b[1]], [a[2], b[2]], color = (:gray, 0.75), linewidth = 1)
    end
    poly!(ax, [Point2f(c1...), Point2f(v1...), Point2f(c2...), Point2f(v2...)],
          color = (:orange, 0.28), strokecolor = :orange, strokewidth = 1.5)
    lines!(ax, [c1[1], c2[1]], [c1[2], c2[2]], color = :crimson,   linewidth = 3)
    lines!(ax, [v1[1], v2[1]], [v1[2], v2[2]], color = :royalblue, linewidth = 3)
    scatter!(ax, [x0[1]], [x0[2]], color = :black, markersize = 9)
    hidedecorations!(ax)
end
fig

## 3. The one algebraic fact

$\Pi_eA_e=\mathrm{Id}$. Everything downstream is a corollary:

* the lift is **exact** on every $u\in[P_k]^2$, hence it satisfies the star property;
* the sandwich reproduces $\int_{D_e}u\cdot Ku$ **exactly** for the same $u$;
* the stabilisation obeys $(I-Q_e)A_e=0$, so it is invisible to reduced polynomials —
  which is why it can never break consistency, and equally why no amount of it can give
  the sandwich the star property.

The `rescale` keyword divides the basis by $s^{|\alpha|}$; it cancels identically in
$\Pi'M\Pi$ and in the lift weights, so it is a pure conditioning device.

In [ ]:
for ol in (true, false)
    ex, cnd = test_lift_exactness(mesh; only_linear = ol)
    println(ol ? "[P1]^2 (N=6) " : "[P2]^2 (N=12)",
            "   max ||Pi A - Id|| = ", ex, "     max cond(A) = ", cnd)
end

println()
for ol in (true, false)
    r = test_local_exactness(mesh; only_linear = ol)
    println(ol ? "[P1]^2" : "[P2]^2",
            "   lift (STAR) ",      r.lift_rel,
            "   sandwich energy ",  r.energy_rel,
            "   (I-Q)A ",           r.stab_rel)
end

## 4. Build both stars

Four operators: each star at each order. Note the last column — the sandwich is
symmetric *by construction*, on any mesh; the lift's antisymmetry is a few percent and
does not shrink with $h$.

In [ ]:
@time Hs1 = hodge_star_sandwich_matrix(mesh; only_linear = true);
@time Hl1 = hodge_star_lift_matrix(mesh;     only_linear = true);
@time Hs2 = hodge_star_sandwich_matrix(mesh; only_linear = false);
@time Hl2 = hodge_star_lift_matrix(mesh;     only_linear = false);

Hdiag = sparse(star(mesh, 1, true))          # the reference: |*e| / |e|

println()
println(rpad("star", 18), rpad("nnz/row", 10), "||H - H'|| / ||H||")
for (nm, H) in (("diagonal", Hdiag), ("sandwich [P1]", Hs1), ("lift [P1]", Hl1),
                ("sandwich [P2]", Hs2), ("lift [P2]", Hl2))
    println(rpad(nm, 18), rpad(round(nnz(H)/size(H,1), digits = 1), 10),
            round(norm(H - H')/norm(H), sigdigits = 3))
end

## 5. The star property — the decisive test

A Hodge star is a **map**: applied to the reduction of $u$, it must return the reduction
of $\star_Ku$ on the dual edge,

$$H\,\mathcal R_1(u)\;=\;\tilde{\mathcal R}_1(\star_K u).$$

A **constant** field is the sharpest probe: it is periodic (so no periodic-seam
artefact) and a genuine star must be exact on it. The lift is, to $10^{-16}$. The
sandwich is not — and what it is off by is not a discretisation error.

In [ ]:
vsm = x -> [sin(2pi*x[2]/mesh.y_period), cos(2pi*x[1]/mesh.x_period)]

println(rpad("star", 18), rpad("constant field", 24), "smooth periodic field")
for (nm, H) in (("diagonal", Hdiag), ("sandwich [P1]", Hs1), ("lift [P1]", Hl1),
                ("sandwich [P2]", Hs2), ("lift [P2]", Hl2))
    println(rpad(nm, 18),
            rpad(round(star_defect_constant(mesh, H), sigdigits = 3), 24),
            round(test_star_property(mesh, H, vsm), sigdigits = 3))
end

### 5b. …and refining does not help

The lift's defect on a constant stays at machine precision; the sandwich's stays at a few
percent. That flatness is the whole point: the sandwich's $\delta$ is an $O(1)$ property
of the Gram form, not an error a finer mesh removes. A Gram matrix is a *bilinear form*
and a star is a *map*; making the sandwich SPD does not change which of the two it is.

In [ ]:
meshfiles = filter(isfile, ["meshes/mesh$(i)_regular.nc" for i in 1:4])

println(rpad("mesh", 24), rpad("cells", 8), rpad("h", 12),
        rpad("sand P1", 12), rpad("sand P2", 12), rpad("lift P1", 12), "lift P2")
for mf in meshfiles
    m  = VoronoiMesh(mf)
    hh = maximum(edge_lengths(m, true))
    s1 = star_defect_constant(m, hodge_star_sandwich_matrix(m; only_linear = true))
    s2 = star_defect_constant(m, hodge_star_sandwich_matrix(m; only_linear = false))
    l1 = star_defect_constant(m, hodge_star_lift_matrix(m;     only_linear = true))
    l2 = star_defect_constant(m, hodge_star_lift_matrix(m;     only_linear = false))
    println(rpad(basename(mf), 24), rpad(m.cells.n, 8), rpad(round(hh, sigdigits = 8), 12),
            rpad(round(s1, sigdigits = 3), 12), rpad(round(s2, sigdigits = 8), 12),
            rpad(round(l1, sigdigits = 2), 12), round(l2, sigdigits = 8))
end

## 6. Eigenvalues — the headline

Generalised problem $L\,p=\lambda\,\star_0 p$ with $L=d_0'Hd_0$ and $\star_0=\mathrm{diag}|c|$.
On a square torus of period $P$ the first non-zero eigenvalue is $4\pi^2/P^2$ with
multiplicity 4.

This is the answer to *"the lift does not give the right eigenvalues"*: it is an **order**
problem, not a symmetry problem. With `only_linear=true` the lift Laplacian is violently
indefinite and the negative part grows like $h^{-2}$. With `only_linear=false` — the same
code, one keyword — there are no negative eigenvalues, the low spectrum is real, and the
eigenvalues converge to $4\pi^2/P^2$.

Widening the stencil at fixed order makes it *worse*, so extra least-squares averaging is
not the cure. Symmetrising is not the cure either:
$\mathrm{sym}(d_0'Hd_0)=d_0'\,\mathrm{sym}(H)\,d_0$, so symmetrising the Laplacian *is*
symmetrising the star — and $\mathrm{sym}(H_{\rm lift})$ loses the star property while
moving the eigenvalues only in the far digits.

In [ ]:
S0     = Matrix(star(mesh, 0, true))            # Diagonal(|Voronoi cell|)
lam_ex = 4pi^2 / mesh.x_period^2                # exact lambda_1, multiplicity 4

function spectrum(H; sym::Bool)
    L = Matrix(laplacian_primal(mesh, H))
    if sym
        return complex.(eigvals(Symmetric(0.5*(L + L')), Symmetric(S0)))
    else
        ev = eigvals(L, S0)
        return ev[sortperm(real.(ev))]
    end
end

println("exact lambda_1 = ", lam_ex, "   (multiplicity 4)")
println()
println(rpad("star", 18), rpad("lambda_1 .. lambda_4", 44),
        rpad("# neg", 8), rpad("# cplx pairs", 14), "max |Im/Re|")
for (nm, H, sy) in (("diagonal", Hdiag, true), ("sandwich [P1]", Hs1, true),
                    ("lift [P1]", Hl1, false), ("sandwich [P2]", Hs2, true),
                    ("lift [P2]", Hl2, false))
    ev  = spectrum(H; sym = sy)
    re  = real.(ev)
    neg = count(x -> x < -1e-8 * maximum(re), re)
    cp  = count(x -> abs(imag(x)) > 1e-10, ev) / 2
    ri  = maximum(abs.(imag.(ev)) ./ max.(abs.(re), 1e-30))
    println(rpad(nm, 18), rpad(string(round.(re[2:5], digits = 3)), 44),
            rpad(neg, 8), rpad(Int(floor(cp)), 14), round(ri, sigdigits = 3))
end

The same thing as a picture: where the lift's spectrum actually sits in the complex
plane. Anything left of the red line is a mode the Laplacian has got backwards.

In [ ]:
fig = Figure(size = (880, 340))
for (i, (nm, H)) in enumerate((("lift  [P1]^2   (only_linear = true)",  Hl1),
                               ("lift  [P2]^2   (only_linear = false)", Hl2)))
    ev  = spectrum(H; sym = false)
    re  = real.(ev)
    neg = count(x -> x < -1e-8 * maximum(re), re)
    ax  = Axis(fig[1, i], xlabel = "Re lambda", ylabel = "Im lambda",
               title = nm * "   (# negative = " * string(neg) * ")")
    scatter!(ax, re, imag.(ev), markersize = 5, color = (:royalblue, 0.7))
    vlines!(ax, [0.0], color = :red, linestyle = :dash)
end
fig

## 7. Poisson convergence — the primal (energy) form

$-\Delta p=f$ solved as $L\,p=\tilde{\mathcal R}_2 f$ with $L=d_0'H_{\rm sandwich}d_0$.

Two honest remarks:

* the right-hand side is integrated over the Voronoi cells with a degree-5 rule
  (`discretize_scalar_field_cells`). The midpoint rule $|c|\,f(x_c)$ is $O(h^2)$ and
  would cap every rate at 2 — the star would then get blamed for the quadrature;
* at **lowest** order the sandwich is *worse* than the plain diagonal star. The
  $[P_1]^2$ sandwich buys nothing. The gain is entirely at $[P_2]^2$.

In [ ]:
pex(m, x)  = sin(2pi*x[1]/m.x_period) * sin(2pi*x[2]/m.y_period)
fex(m, x)  = (4pi^2/m.x_period^2 + 4pi^2/m.y_period^2) * pex(m, x)
gpex(m, x) = [2pi/m.x_period * cos(2pi*x[1]/m.x_period) * sin(2pi*x[2]/m.y_period),
              2pi/m.y_period * sin(2pi*x[1]/m.x_period) * cos(2pi*x[2]/m.y_period)]

function poisson_error(m, H)
    p  = solve_poisson_primal(m, H, x -> fex(m, x))
    pe = [pex(m, m.cells.position[c]) for c in 1:m.cells.n]
    pe .-= sum(pe)/length(pe)
    a  = m.cells.area
    return sqrt(sum(a .* (p .- pe).^2) / sum(a .* pe.^2))
end

hs   = Float64[]
errs = Dict(k => Float64[] for k in ("diagonal", "sandwich [P1]", "sandwich [P2]"))
for mf in meshfiles
    m = VoronoiMesh(mf)
    push!(hs, maximum(edge_lengths(m, true)))
    push!(errs["diagonal"],      poisson_error(m, sparse(star(m, 1, true))))
    push!(errs["sandwich [P1]"], poisson_error(m, hodge_star_sandwich_matrix(m; only_linear = true)))
    push!(errs["sandwich [P2]"], poisson_error(m, hodge_star_sandwich_matrix(m; only_linear = false)))
end

rates(e) = [log(e[i]/e[i+1]) / log(hs[i]/hs[i+1]) for i in 1:length(e)-1]
for k in ("diagonal", "sandwich [P1]", "sandwich [P2]")
    println(rpad(k, 16), " err ", round.(errs[k], sigdigits = 3),
            "   rates ", round.(rates(errs[k]), digits = 2))
end

In [ ]:
fig = Figure(size = (540, 420))
ax  = Axis(fig[1, 1], xscale = log10, yscale = log10,
           xlabel = "h", ylabel = "relative L2 error in p", title = "primal Poisson")
for (k, c) in (("diagonal", :black), ("sandwich [P1]", :crimson), ("sandwich [P2]", :royalblue))
    scatterlines!(ax, hs, errs[k], color = c, label = k)
end
lines!(ax, hs, errs["diagonal"][1] .* (hs ./ hs[1]).^2, color = (:gray, 0.6),
       linestyle = :dash, label = "h^2")
lines!(ax, hs, errs["sandwich [P2]"][1] .* (hs ./ hs[1]).^3, color = (:gray, 0.6),
       linestyle = :dot, label = "h^3")
axislegend(ax, position = :rb)
fig

## 8. Flux — the star on its own

Applying the lift to the **exact** reduced gradient isolates the star from the solve.

Worth noticing first: $d_0\mathcal R_0p=\mathcal R_1(\nabla p)$ is exact for a
straight edge. The column below is *not* zero on coarse meshes because `edge_integral`
uses 4-point Gauss on a trigonometric $\nabla p$ (quadrature error, not $d_0$).
The identity $\Pi_e A_e=I$ is zero; it is a different statement and is not what that
column measures.

Recovering the flux from a *computed* potential, $H_{\rm lift}\,d_0\,p_h$, caps the rate
at the potential's, because differentiating $p_h-p$ costs one power of $h$. For the
high-order flux directly, solve the mixed system in §9.

In [ ]:
println(rpad("mesh", 22), rpad("h", 12), rpad("lift [P1]", 14), rpad("lift [P2]", 14),
        "|| d0 R0 p - R1 grad p ||")
el1, el2, hs8 = Float64[], Float64[], Float64[]
for mf in meshfiles
    m  = VoronoiMesh(mf)
    hh = maximum(edge_lengths(m, true));  push!(hs8, hh)
    e1 = test_star_property(m, hodge_star_lift_matrix(m; only_linear = true),  x -> gpex(m, x))
    e2 = test_star_property(m, hodge_star_lift_matrix(m; only_linear = false), x -> gpex(m, x))
    push!(el1, e1); push!(el2, e2)
    p0  = [pex(m, m.cells.position[c]) for c in 1:m.cells.n]
    ftc = maximum(abs.(d0_matrix_primal(m) * p0 .- reduce_primal_edges(m, x -> gpex(m, x)).values))
    println(rpad(basename(mf), 22), rpad(round(hh, sigdigits = 3), 12),
            rpad(round(e1, sigdigits = 3), 14), rpad(round(e2, sigdigits = 3), 14), ftc)
end
println()
println("lift [P1] rates : ", round.([log(el1[i]/el1[i+1])/log(hs8[i]/hs8[i+1]) for i in 1:length(el1)-1], digits = 2))
println("lift [P2] rates : ", round.([log(el2[i]/el2[i+1])/log(hs8[i]/hs8[i+1]) for i in 1:length(el2)-1], digits = 2))

## 9. The mixed system — Hirani–Nakshatrala–Chaudhry, eq. (4.6)

$$\begin{bmatrix}-\mu M_1 & D_1^{\top}\\ D_1 & 0\end{bmatrix}\begin{bmatrix}f\\p\end{bmatrix}=\begin{bmatrix}0\\ \varphi\end{bmatrix}$$

with $f$ a primal 1-cochain on Delaunay edges, $p$ a dual 0-cochain at the circumcentres,
$D_1=$ `d1_matrix_primal`, and $M_1$ **the lift star** built with $K=\kappa^{-1}$.

This is the arrangement in which the star sits inside an *equation* whose residual must
lie in $\mathrm{range}(D_1^{\top})$ — the patch test, exactly what the sandwich fails.
Use the lift here.

On a torus the system is singular twice over: the constant in $p$, and the two harmonic
1-cochains of $H^1(\mathbb{T}^2)$. `pinv` takes the minimum-norm solution, which is the
one orthogonal to the harmonics — the right choice for a source with zero mean flux. The
overall sign of $f$ follows `d1_matrix_primal`'s orientation convention, so both are
reported.

`pinv` on the dense saddle-point matrix is $O(n^3)$ — fine for `mesh1`/`mesh2`, but on a
large mesh replace it with an iterative saddle-point solver (MINRES on the symmetrised
system, or Uzawa) and deflate the three-dimensional null space explicitly.

In [ ]:
M1       = hodge_star_lift_matrix(mesh; only_linear = false)
Amat, D1 = darcy_mixed_system(mesh, M1)
ne, nt   = mesh.edges.n, mesh.vertices.n

phi  = discretize_scalar_field_triangles_ho(mesh, x -> fex(mesh, x)).values
phi .-= sum(phi)/length(phi)                       # compatibility on a torus
sol   = pinv(Matrix(Amat)) * [zeros(ne); phi]      # minimum-norm: kills the harmonics
f_h   = sol[1:ne]
p_h   = sol[ne+1:end];  p_h .-= sum(p_h)/length(p_h)

f_ex   = reduce_primal_edges(mesh, x -> -gpex(mesh, x)).values   # f = -grad p, kappa = I
lp     = edge_lengths(mesh, true);  ld = edge_lengths(mesh, false)
w      = lp ./ ld
nrm(v) = sqrt(sum(w .* v.^2))
println("flux error (+f) : ", nrm(f_h .- f_ex) / nrm(f_ex))
println("flux error (-f) : ", nrm(f_h .+ f_ex) / nrm(f_ex))

p_ex  = [pex(mesh, mesh.vertices.position[k]) for k in 1:nt];  p_ex .-= sum(p_ex)/nt
a_t   = mesh.vertices.area
println("pressure error  : ", min(sqrt(sum(a_t .* (p_h .- p_ex).^2)),
                                  sqrt(sum(a_t .* (p_h .+ p_ex).^2))) /
                              sqrt(sum(a_t .* p_ex.^2)))

## 10. Everything at once

`report(mesh)` runs all of the above on any mesh and prints it in one block. Run it on a
sequence of refined meshes and watch which numbers move and which do not.

In [ ]:
report(mesh);

## 11. Anisotropic $\kappa$

`K` is any function returning a symmetric $2\times2$ matrix. It goes into the mass matrix
for the sandwich and into the continuous star for the lift, so the two stay consistent.

Put $K=\kappa$ for the energy form $\int\kappa\,\nabla p\cdot\nabla p$, and
$K=\kappa^{-1}$ for the Darcy/mixed form $\int\kappa^{-1}f\cdot f$.

In [ ]:
# NB: never write `[2.0 + sin(u)  0.3; ...]` -- inside [ ] the spaces around `+`
# are parse-ambiguous in Julia.  Build the entries first.
function kap(x)
    k11 = 2.0 + sin(2pi*x[1]/mesh.x_period)
    return [k11 0.3; 0.3 1.0]
end
ikap(x) = inv(kap(x))          # <- this is the one for the mixed/Darcy form

Hs_k = hodge_star_sandwich_matrix(mesh; only_linear = false, K = kap)
Hl_k = hodge_star_lift_matrix(mesh;     only_linear = false, K = kap)

println("sandwich, K:  symmetry ", round(norm(Hs_k - Hs_k')/norm(Hs_k), sigdigits = 3),
        "   star defect on a constant ", round(star_defect_constant(mesh, Hs_k; K = kap), sigdigits = 3))
println("lift,     K:  symmetry ", round(norm(Hl_k - Hl_k')/norm(Hl_k), sigdigits = 3),
        "   star defect on a constant ", round(star_defect_constant(mesh, Hl_k; K = kap), sigdigits = 3))

got, exact, rel = test_energy(mesh, Hs_k, x -> [1.0, 0.7]; K = kap)
println("sandwich energy of a constant: ", got, "  vs  ", exact, "     rel ", rel)

## Recommended configuration

```julia
potential / eigenvalues :  hodge_star_sandwich_matrix(mesh; only_linear = false)
flux, recovered         :  hodge_star_lift_matrix(mesh;     only_linear = false)
flux, solved directly   :  darcy_mixed_system(mesh, that lift star)
```

and never the $[P_1]^2$ lift inside a Laplacian.

**One thing that will bite you if you edit the stabilisation.** The DOF rescaling has to
multiply the projector on the *left*, $Z=D^{-1}(I-Q)$, never $(I-Q)D^{-1}$. The wrong
side destroys $(I-Q)A=0$, and "the energy is exact on polynomials" degrades from
$4\times10^{-15}$ to $10^{-4}$ — small enough to look like quadrature error and never be
questioned. That bug was in my first draft of this file; `test_local_exactness` is what
caught it, so run it after any change.